# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides users through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities via their `@id` fields.

### Dataset Source
The dataset is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
First, load metadata and records from the FAIR^2 dataset using `mlcroissant`. This provides structured access to all entities defined by their `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript or iterate)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All exploration will reference these unique identifiers.

**Note**: The Croissant schema defines all entities with `@id`. Here we list record sets and sample fields.

In [ ]:
# List all record sets by @id
record_sets = []

# Check if dataset has recordSet property
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"RecordSet @id: {rs['@id']}")
        record_sets.append(rs['@id'])
else:
    print("No record sets defined in this schema. Fetching automatic record sets from mlcroissant if available...")
    # Try to discover record sets dynamically
    record_sets = dataset.list_record_sets()
    for rsid in record_sets:
        print(f"RecordSet @id: {rsid}")

# Preview the first records from each record set (using @id)
for record_set_id in record_sets:
    print(f"\nRecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    for rec in records[:2]:  # Print first two records
        print(rec)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, referencing the record set and field `@id`s. We show how to extract records for all discovered record sets.

In [ ]:
# Collect record sets and their records as DataFrames
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

# Choose the first available record set for subsequent steps
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df_main = dataframes[main_record_set_id]
else:
    main_record_set_id = None
    df_main = None

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records by a numeric field (`@id`), normalize, and group by a categorical field (`@id`). If these fields exist, use them accordingly.

In [ ]:
# Example: Filter, normalize, and group
if df_main is not None:
    # Identify numeric fields by @id
    numeric_fields = df_main.select_dtypes(include='number').columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")
        threshold = df_main[numeric_field_id].mean()
        filtered_df = df_main[df_main[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by categorical field (with @id)
        cat_fields = df_main.select_dtypes(include='object').columns.tolist()
        cat_fields_ids = [f for f in cat_fields if f != numeric_field_id]
        if cat_fields_ids:
            group_field_id = cat_fields_ids[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field found for filtering.")
else:
    print("No DataFrame extracted; cannot perform EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to a categorical variable, referencing by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and numeric_fields:
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.show()

    if cat_fields_ids:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
        plt.title(f"{numeric_field_id} (@id) by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`. We loaded metadata, extracted records, performed basic EDA, and visualized data. For more advanced analyses, refer to the variable and field `@id` definitions in the Croissant schema and FAIR^2 documentation.

**Key Points:**
- All entities (record sets, fields, columns) referenced by `@id` as required.
- Data loading, overview, extraction, and processing steps are extensible to your workflow.
- Use `mlcroissant`'s metadata mapping to ensure reproducibility and clarity in references.